In [1]:
from ultralytics import YOLO

model = YOLO("runs/detect/yolo11n-visdrone-agnostic/training/weights/best.pt")

In [2]:
model.train(
    data="DIOR/dior_filtered_classes/data.yaml", 
    imgsz=1280, 
    epochs=150, 
    dropout=0.2,
    project="yolo11n-visdrone-agnostic",
    name="10-few-shot",
    resume=True)

New https://pypi.org/project/ultralytics/8.4.38 available 😃 Update with 'pip install -U ultralytics'
WARNING ⚠️ model 'runs/detect/yolo11n-visdrone-agnostic/training/weights/best.pt' is not a resumable training checkpoint (missing epoch/optimizer state). Use 'resume' only to continue incomplete training. Starting new training instead.
Ultralytics 8.4.37 🚀 Python-3.11.7 torch-2.11.0+cu130 CUDA:0 (NVIDIA A100 80GB PCIe, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=DIOR/dior_filtered_classes/data.yaml, degrees=0.0, deterministic=True, device=3, dfl=1.5, dnn=False, dropout=0.2, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x794b401a5510>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [3]:
results = model.val(data="DIOR/dior_filtered_classes/data.yaml", split='test')

Ultralytics 8.4.37 🚀 Python-3.11.7 torch-2.11.0+cu130 CUDA:0 (NVIDIA A100 80GB PCIe, 81153MiB)
YOLO11n summary (fused): 101 layers, 2,582,347 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 5652.2±1511.7 MB/s, size: 516.9 KB)
val: Scanning /home/gpu_03/class-agnostic/DIOR/dior_filtered_classes/test/labels.cache... 679 images, 0 backgrounds, 550 corrupt: 100% ━━━━━━━━━━━━ 679/679 237.3Mit/s 0.0s
val: /home/gpu_03/class-agnostic/DIOR/dior_filtered_classes/test/images/00007.jpg: ignoring corrupt image/label: Label class 7 exceeds dataset class count 1. Possible class labels are 0-0
val: /home/gpu_03/class-agnostic/DIOR/dior_filtered_classes/test/images/00011.jpg: ignoring corrupt image/label: Label class 8 exceeds dataset class count 1. Possible class labels are 0-0
val: /home/gpu_03/class-agnostic/DIOR/dior_filtered_classes/test/images/00075.jpg: ignoring corrupt image/label: Label class 7 exceeds dataset class count 1. Possible class labels are 0-0


In [ ]:
from ultralytics import YOLO
import os
import cv2
import numpy as np
from pathlib import Path

# Paths
img_path = 'DIOR/dior_filtered_classes/test/images'
label_path = 'DIOR/dior_filtered_classes/test/labels'
model_path = 'runs/detect/yolo11n-visdrone-agnostic/10-few-shot/weights/best.pt'
save_dir = 'runs/detect/yolo11n-visdrone-agnostic/10-few-shot/visualization'

# Create save directory if it doesn't exist
os.makedirs(save_dir, exist_ok=True)

# Load YOLO model
model = YOLO(model_path)

def convert_yolo_to_bbox(yolo_coords, img_width, img_height):
    """
    Convert YOLO format (cx, cy, w, h) normalized to absolute coordinates
    """
    cx, cy, w, h = yolo_coords
    x1 = int((cx - w/2) * img_width)
    y1 = int((cy - h/2) * img_height)
    x2 = int((cx + w/2) * img_width)
    y2 = int((cy + h/2) * img_height)
    # Clamp to image boundaries
    x1 = max(0, min(x1, img_width))
    x2 = max(0, min(x2, img_width))
    y1 = max(0, min(y1, img_height))
    y2 = max(0, min(y2, img_height))
    return x1, y1, x2, y2

# Get all image files
image_files = [f for f in os.listdir(img_path) if f.endswith(('.jpg', '.jpeg', '.png'))]

print(f"Found {len(image_files)} images")
print(f"Results will be saved to: {save_dir}")

for img_idx, img_file in enumerate(image_files):
    # Load image
    img_path_full = os.path.join(img_path, img_file)
    image = cv2.imread(img_path_full)
    
    if image is None:
        print(f"Warning: Could not read {img_path_full}")
        continue
    
    img_height, img_width = image.shape[:2]
    
    # 1. Draw GROUND TRUTH boxes (GREEN) - show only class number
    label_file = os.path.join(label_path, Path(img_file).stem + '.txt')
    gt_count = 0
    
    if os.path.exists(label_file):
        with open(label_file, 'r') as f:
            for line in f.readlines():
                parts = line.strip().split()
                if len(parts) >= 5:
                    class_id = int(parts[0])
                    yolo_coords = [float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])]
                    x1, y1, x2, y2 = convert_yolo_to_bbox(yolo_coords, img_width, img_height)
                    
                    # Draw green rectangle (thickness 2)
                    cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)
                    
                    # Label: only class number
                    label = f"{class_id}"
                    
                    # Calculate text size for background
                    (label_w, label_h), baseline = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
                    
                    # Draw background rectangle for text
                    # cv2.rectangle(image, (x1, y1 - label_h - 5), (x1 + label_w, y1), (0, 255, 0), -1)
                    
                    # Draw text (black color on green background)
                    # cv2.putText(image, label, (x1, y1 - 3), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2)
                    
                    gt_count += 1
    
    # 2. Draw PREDICTIONS (BLUE) - show only confidence
    results = model(img_path_full)
    pred_count = 0
    
    if len(results) > 0 and results[0].boxes is not None:
        for box in results[0].boxes:
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
            confidence = box.conf[0]
            
            # Draw blue rectangle (thickness 2)
            cv2.rectangle(image, (x1, y1), (x2, y2), (255, 0, 0), 2)
            
            # Label: only confidence (rounded to 2 decimal places)
            label = f"{confidence:.2f}"
            
            # Calculate text size for background
            (label_w, label_h), baseline = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2)
            
            # Draw background rectangle for text (above the box)
            # y_text = y1 - 5
            # if y_text - label_h < 0:  # If text would go above image, put it inside the box
            #     y_text = y1 + label_h + 5
            
            # cv2.rectangle(image, (x1, y_text - label_h - 3), (x1 + label_w, y_text), (255, 0, 0), -1)
            
            # Draw text (white color on blue background)
            # cv2.putText(image, label, (x1, y_text - 3), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
            
            pred_count += 1
    
    # # Add legend to the top-left corner of the image
    # legend_y = 30
    # cv2.rectangle(image, (5, 5), (250, 75), (0, 0, 0), -1)  # Black background for legend
    # cv2.rectangle(image, (5, 5), (250, 75), (255, 255, 255), 1)  # White border
    
    # # Green box legend
    # cv2.rectangle(image, (10, legend_y - 10), (30, legend_y + 5), (0, 255, 0), 2)
    # cv2.putText(image, "GT: Class ID", (35, legend_y), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    
    # # Blue box legend
    # cv2.rectangle(image, (10, legend_y + 20), (30, legend_y + 35), (255, 0, 0), 2)
    # cv2.putText(image, "Pred: Confidence", (35, legend_y + 32), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    
    # # Add counts to legend
    # cv2.putText(image, f"GT: {gt_count}  |  Pred: {pred_count}", 
    #             (10, legend_y + 55), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 200, 200), 1)
    
    # Save the result
    save_path = os.path.join(save_dir, img_file)
    cv2.imwrite(save_path, image)
    
    print(f"[{img_idx+1}/{len(image_files)}] {img_file}: {gt_count} GT boxes, {pred_count} predictions")

print(f"\n✓ Visualization complete!")
print(f"✓ Results saved to: {save_dir}")
print(f"\nLegend:")
print(f"  🟢 Green boxes: Ground Truth (shows class number only)")
print(f"  🔵 Blue boxes: Predictions (shows confidence score only)")

Found 679 images
Results will be saved to: runs/detect/yolo11n-visdrone-agnostic/10-few-shot/visualization

image 1/1 /home/gpu_03/class-agnostic/DIOR/dior_filtered_classes/test/images/15394.jpg: 1280x1280 16 0s, 4.7ms
Speed: 6.1ms preprocess, 4.7ms inference, 0.9ms postprocess per image at shape (1, 3, 1280, 1280)
[1/679] 15394.jpg: 2 GT boxes, 16 predictions

image 1/1 /home/gpu_03/class-agnostic/DIOR/dior_filtered_classes/test/images/06272.jpg: 1280x1280 12 0s, 4.5ms
Speed: 4.9ms preprocess, 4.5ms inference, 0.8ms postprocess per image at shape (1, 3, 1280, 1280)
[2/679] 06272.jpg: 8 GT boxes, 12 predictions

image 1/1 /home/gpu_03/class-agnostic/DIOR/dior_filtered_classes/test/images/09527.jpg: 1280x1280 1 0, 4.0ms
Speed: 4.6ms preprocess, 4.0ms inference, 0.8ms postprocess per image at shape (1, 3, 1280, 1280)
[3/679] 09527.jpg: 2 GT boxes, 1 predictions

image 1/1 /home/gpu_03/class-agnostic/DIOR/dior_filtered_classes/test/images/22522.jpg: 1280x1280 (no detections), 3.9ms
Speed: